In [ ]:
import numpy as np
import pandas as pd
import dai


def main(datasources, start_date, end_date):
    """
    F39 / F27_a: failed intraday breakout crowded exhaustion.

    Stocks with concentrated trading, strong intraday high, but weak close
    position often reflect short-term crowding that failed to hold. Higher
    factor means less exhaustion risk. The final factor is shifted by one
    trading day to avoid lookahead when used as a next-day predictor.
    """
    table_name = datasources["bar1m"]
    query_start = (pd.to_datetime(start_date) - pd.Timedelta(days=20)).strftime("%Y-%m-%d %H:%M:%S")

    daily = dai.query(
        f"""
        SELECT
            date::DATE::DATETIME AS date,
            instrument,
            FIRST(CAST(open AS DOUBLE) ORDER BY date) AS open,
            MAX(CAST(high AS DOUBLE)) AS high,
            MIN(CAST(low AS DOUBLE)) AS low,
            LAST(CAST(close AS DOUBLE) ORDER BY date) AS close,
            SUM(CAST(amount AS DOUBLE)) AS total_amount,
            SUM(CAST(amount AS DOUBLE) * CAST(amount AS DOUBLE)) AS amount_sq_sum
        FROM {table_name}
        GROUP BY date::DATE, instrument
        ORDER BY date, instrument
        """,
        filters={"date": [query_start, end_date]},
        compression=True,
    ).df()

    daily["date"] = pd.to_datetime(daily["date"]).dt.normalize()
    for col in ["open", "high", "low", "close", "total_amount", "amount_sq_sum"]:
        daily[col] = pd.to_numeric(daily[col], errors="coerce")

    daily["concentration"] = np.where(
        daily["total_amount"] > 0,
        daily["amount_sq_sum"] / (daily["total_amount"] ** 2),
        np.nan,
    )
    daily["intraday_up"] = np.where(daily["open"] > 0, daily["high"] / daily["open"] - 1.0, np.nan)
    daily["close_position"] = np.where(
        daily["high"] > daily["low"],
        (daily["close"] - daily["low"]) / (daily["high"] - daily["low"]),
        np.nan,
    )

    daily["concentration_rank"] = daily.groupby("date")["concentration"].rank(pct=True)
    daily["intraday_up_rank"] = daily.groupby("date")["intraday_up"].rank(pct=True)
    daily["failed_close_rank"] = daily.groupby("date")["close_position"].rank(pct=True, ascending=False)

    daily["raw_factor"] = -(
        0.45 * daily["concentration_rank"]
        + 0.35 * daily["intraday_up_rank"]
        + 0.20 * daily["failed_close_rank"]
    )
    daily = daily.sort_values(["instrument", "date"]).copy()
    daily["factor"] = daily.groupby("instrument")["raw_factor"].shift(1)

    factor = daily[["date", "instrument", "factor"]].copy()
    factor = factor[
        (factor["date"] >= pd.to_datetime(start_date).normalize())
        & (factor["date"] <= pd.to_datetime(end_date).normalize())
    ]
    factor["factor"] = pd.to_numeric(factor["factor"], errors="coerce")

    stk_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [start_date, end_date]},
    ).df()
    stk_pool["date"] = pd.to_datetime(stk_pool["date"]).dt.normalize()

    return (
        pd.merge(factor, stk_pool, how="inner", on=["date", "instrument"])
        .dropna(subset=["factor"])
        .sort_values(["date", "instrument"])
        .reset_index(drop=True)
        .loc[:, ["date", "instrument", "factor"]]
    )
